# Step 1 — immutable Kaggle T4 x2 preflight

This notebook launches only the standard-stack plumbing preflight: a fixed real-batch diagnostic, one standard Trainer checkpoint, a fresh Trainer resume, a two-rank versus one-process loss-normalization comparison, and a standard Hugging Face artifact round trip.

The checked-in file is a renderable template. Generate a local, SHA-pinned launcher after committing the runner with `python step1/kaggle/render_preflight_notebook.py --commit <final-40-char-sha>`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/escher-bach/actuallybuildingstuff.git'
GIT_COMMIT = '__FINAL_COMMIT_SHA__'
CONFIG_REL = 'step1/configs/kaggle/t4x2_preflight.toml'

if GIT_COMMIT == '__FINAL_COMMIT_SHA__':
    raise RuntimeError('Generate this template with render_preflight_notebook.py after committing the runner.')
assert len(GIT_COMMIT) == 40 and all(c in '0123456789abcdef' for c in GIT_COMMIT)
WORKING = Path('/kaggle/working')
SOURCE = WORKING / 'actuallybuildingstuff'
PROJECT = SOURCE / 'baby-llm-foundations'
OUTPUT = WORKING / 'step1-results'
assert not SOURCE.exists(), f'fresh batch session required; already exists: {SOURCE}'
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
env = os.environ.copy()
env.update({'GIT_TERMINAL_PROMPT': '0', 'PYTHONUNBUFFERED': '1', 'PIP_DISABLE_PIP_VERSION_CHECK': '1', 'WANDB_MODE': 'disabled', 'TOKENIZERS_PARALLELISM': 'false'})
subprocess.run(['git', 'clone', REPO_URL, str(SOURCE)], check=True, env=env)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', GIT_COMMIT], check=True, env=env)
resolved = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True, env=env).strip()
assert resolved == GIT_COMMIT, (resolved, GIT_COMMIT)
assert (PROJECT / CONFIG_REL).is_file(), PROJECT / CONFIG_REL


In [ ]:
cmd = [sys.executable, '-m', 'step1_experiments.runner', '--config', str(PROJECT / CONFIG_REL), '--output-root', str(OUTPUT), '--resume', 'auto']
completed = subprocess.run(cmd, cwd=str(PROJECT / 'step1' / 'python'), env=env, check=False)
if completed.returncode != 0:
    raise RuntimeError(f'Step 1 runner failed with exit code {completed.returncode}; download the failure bundle from {OUTPUT}')


In [ ]:
import json
reports = sorted(OUTPUT.glob('*/preflight_report.json'))
assert len(reports) == 1, reports
report = json.loads(reports[0].read_text())
assert report['ranks_finished'] == [0, 1], report
assert report['resumed_global_step'] == report['checkpoint_after_updates'] + report['resume_updates'], report
assert report['fixed_real_batch_loss']['after_resume'] < report['fixed_real_batch_loss']['initial'], report
print(json.dumps(report, indent=2, sort_keys=True))
